# SALM Fine-tuning 11 · Forty-five languages, one guess: spoken language ID (CommonLanguage)

> **Task** clip → which of 45 languages is spoken · **Data** CommonLanguage (Common Voice subset)
> **Metric** accuracy ↑ · **Result** 49.81% → **92.87%** (+86.5%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: it can transcribe them, but can it name them?

CommonLanguage has short read sentences in 45 languages, from Arabic to Welsh, including some you may not have met:
Chuvash, Dhivehi, Hakha Chin, Kabyle, Sakha, Romansh Sursilvan… Speakers never appear in two splits. With the list of
45 in the prompt, the base model gets ~50%. Big languages are fine; small ones get confused with their neighbors. The
goal is a 45-way language identifier that works on unseen speakers.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: fix the labels, then shuffle

The dataset's labels include typos (`Mangolian`, `Ukranian`) and underscores (`Chinese_Hongkong`). The prep script
uses correctly spelled names, so a model that writes "Ukrainian" isn't marked wrong. The mirror also declares 48 kHz
audio that is really 16 kHz (the prep script reads the file header).

In [ ]:
d = M / "commonlanguage"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py commonlanguage", log="prep_lid.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
print(open(d / "labels.txt").read().split("\n"))

## 2. Baseline

In [ ]:
TASK = f"--task cls --max-tokens 16 --labels {d}/labels.txt"
P2 = "{context} Reply with the language name only."
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"lid_base_{p}_{s}", TASK, extra)["score"]
    print(f"{p}  dev {base[p, 'dev']:5.2f}%   test {base[p, 'test']:5.2f}%")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = max(base[p, "test"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")  # train with p2: it pins the answer format

## 3. The most important line in this notebook

The training manifest is **sorted by language**: all Arabic, then all Basque, and so on. Lhotse shuffles through a
10,000-row buffer, which can't mix a 22,194-row sorted file. Without a shuffle on disk, training sees a few languages
at a time (the loss oscillates, e.g. 0.17 → 0.51 → 0.16 → 0.64) and the model collapses to answering one language
(for example **"Maltese"**) for almost every clip, at ~3% accuracy. Shuffle on disk:

In [ ]:
def label_change_rate(path):
    labels = [json.loads(l)["text"] for l in open(path)]
    return sum(a != b for a, b in zip(labels, labels[1:])) / (len(labels) - 1)
print(f"label-change rate before: {label_change_rate(d / 'train_ft.json'):.3f}")
sh(f"{PYTHON} {FT}/shuffle_manifest.py {d}/train_ft.json")
print(f"label-change rate after:  {label_change_rate(d / 'train_ft.json'):.3f}")

## 4. Config and training (~35 min)

In [ ]:
CFG = WORK / "configs" / "lid.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'lid'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name lid_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 450 "
   f"--batch-tokens 32000 --limit-train-batches 90 --limit-val-batches 50 --save-top-k -1")
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 360 450")

## 5. Select on dev

Expected dev accuracy: step 360 → 92.84%, **step 450 → 93.20%**. Step 450 is selected.

In [ ]:
dev = {}
for step in (360, 450):
    out = EXPORT / f"lid_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'lid'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_lid_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"lid_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev accuracy {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 6. Test on unseen speakers, once

In [ ]:
MODEL = EXPORT / f"lid_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "lid_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "lid_librispeech", "--task asr --normalizer english")
FT_ACC = 100 * ft["score"]
print(f"test accuracy: base {BASELINE:.2f}% → fine-tuned {FT_ACC:.2f}% ({(FT_ACC - BASELINE) / BASELINE:+.1%}); LibriSpeech {100 * guard['score']:.2f}%")
result("11 baseline test accuracy", BASELINE, 49.81, 0.5)
result("11 fine-tuned test accuracy", FT_ACC, 92.87, 1.0)

Expected results:

| | test accuracy | LibriSpeech WER |
|---|---:|---:|
| base, prompt chosen on dev (p1) | 49.81% | 1.54% |
| **fine-tuned** | **92.87%** | 1.56% |

**+86.5% relative.** The prompt chosen on dev (p1, the plain list of 45 labels) is also the test-oracle.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="lid")

## Takeaways

1. **Shuffle on disk.** A shuffle buffer is not a shuffle.
2. **A model that answers one class for everything has a data-order bug** until proven otherwise.
3. **Clean label strings** so correct spellings aren't punished.